# 04 — GMM Density Modelling & BIC-Based Component Selection

The classical pipeline models the class-conditional feature density p(x | class) with a **Gaussian Mixture Model (GMM)** fitted via the EM algorithm.  
At inference time the Bayesian decision rule combines these densities with the asymmetric cost matrix to produce the triage decision.

This notebook:
1. Fits GMMs for K ∈ {1, …, 6} on each class's **training** features.
2. Selects K by minimising the **Bayesian Information Criterion (BIC)** — a model-selection score that penalises model complexity to prevent overfitting.
3. Saves the two optimal GMMs for use in the decision step.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.mixture import GaussianMixture

from config import SEED, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR

## 1 — Load training split

In [ ]:
data = np.load(PROCESSED_DIR / "splits.npz")
X_train, y_train = data["X_train"], data["y_train"]

X0 = X_train[y_train == 0]  # benign (nevus)
X1 = X_train[y_train == 1]  # melanoma

print(f"Training set  : {len(X_train)} samples, {X_train.shape[1]} features")
print(f"  Benign      : {len(X0)}")
print(f"  Melanoma    : {len(X1)}")

## 2 — Covariance type choice

A **full** covariance GMM models arbitrary correlations between feature dimensions.  
However, our feature space has 96 dimensions; a single Gaussian with full covariance requires 96 × 97 / 2 = 4 656 free parameters for its covariance matrix alone.  
With a few hundred training samples per class this can easily lead to a **singular (non-invertible) covariance matrix**, causing EM to diverge or produce `-inf` log-likelihoods.

Strategy implemented below:
- **Try `full` first.** If sklearn raises a `ConvergenceWarning` or returns a non-finite BIC, we fall back to `diag`.
- **`diag` covariance** assumes feature dimensions are uncorrelated (diagonal covariance matrix), reducing the parameter count to 96 per component — far more tractable given the dataset size.
- The chosen covariance type is recorded in the summary table and used consistently for both classes.

In [ ]:
K_RANGE = list(range(1, 7))  # [1, 2, 3, 4, 5, 6]
CLASS_DATA = {0: ("Benign",   X0),
              1: ("Melanoma", X1)}

def fit_gmm_bic(X, k_range, seed):
    """Fit GMMs for each K; return (bic_scores, chosen_k, best_model, cov_type)."""
    for cov_type in ("full", "diag"):
        bic_scores = []
        models = []
        ok = True
        for k in k_range:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                gmm = GaussianMixture(
                    n_components=k,
                    covariance_type=cov_type,
                    random_state=seed,
                    max_iter=200,
                    n_init=3,          # 3 random restarts → avoid bad local optima
                )
                gmm.fit(X)
            convergence_issues = any(
                issubclass(w.category, Exception) or "ConvergenceWarning" in str(w.category)
                for w in caught
            )
            bic = gmm.bic(X)
            if not np.isfinite(bic) or convergence_issues:
                ok = False
                break
            bic_scores.append(bic)
            models.append(gmm)

        if ok and len(bic_scores) == len(k_range):
            best_idx  = int(np.argmin(bic_scores))
            return bic_scores, k_range[best_idx], models[best_idx], cov_type

    raise RuntimeError("GMM fitting failed for both 'full' and 'diag' covariance types.")


results = {}
for label, (name, X_cls) in CLASS_DATA.items():
    bic_scores, k_best, best_gmm, cov_type = fit_gmm_bic(X_cls, K_RANGE, SEED)
    results[label] = dict(
        name=name, bic_scores=bic_scores, k_best=k_best,
        best_gmm=best_gmm, cov_type=cov_type
    )
    print(f"{name:10s}: best K={k_best}, BIC={min(bic_scores):.2f}, cov_type='{cov_type}'")

## 3 — BIC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
fig.suptitle("GMM BIC vs. number of components K", fontsize=13)

for ax, (label, res) in zip(axes, results.items()):
    ax.plot(K_RANGE, res["bic_scores"], marker="o", linewidth=2, color="steelblue")
    ax.axvline(res["k_best"], color="tomato", linestyle="--",
               label=f"K={res['k_best']} (min BIC)")
    ax.set_title(f"{res['name']} (cov='{res['cov_type']}')", fontsize=11)
    ax.set_xlabel("K (number of components)")
    ax.set_ylabel("BIC")
    ax.set_xticks(K_RANGE)
    ax.legend()
    ax.grid(True, linestyle=":")

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "bic_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {FIGURES_DIR / 'bic_curves.png'}")

## 4 — Save fitted GMMs

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

gmm_paths = {0: PROCESSED_DIR / "gmm_benign.pkl",
             1: PROCESSED_DIR / "gmm_melanoma.pkl"}

for label, res in results.items():
    joblib.dump(res["best_gmm"], gmm_paths[label])
    print(f"Saved {res['name']} GMM (K={res['k_best']}) → {gmm_paths[label]}")

## 5 — Summary table

In [ ]:
rows = [
    {
        "class":           res["name"],
        "K_chosen":        res["k_best"],
        "BIC_value":       round(min(res["bic_scores"]), 4),
        "covariance_type": res["cov_type"],
    }
    for res in results.values()
]

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TABLES_DIR / "gmm_selection.csv", index=False)
print(f"\nSaved to: {TABLES_DIR / 'gmm_selection.csv'}")